# Reproduce v0.9.3 Validated ODE Microstep

This notebook clones the public repository, verifies the frozen release metadata, recomputes the v0.9.3 Arb certificate, and performs a fail-closed gate check.

In [ ]:
!rm -rf /content/Geometric-Flow
!git clone --depth 1 https://github.com/papasop/Geometric-Flow.git
%cd /content/Geometric-Flow

In [ ]:
!python -m pip install -q -r requirements.txt
!python tools/verify_release.py

In [ ]:
!python src/response_fibre_intrinsic_picard_microstep_v0_9_3.py \
  --inputs-zip inputs/response_fibre_v0_6_2_backend_inputs.zip \
  --v074-source src/response_fibre_arb_kkt_witness_alignment_v0_7_4.py \
  --no-download \
  --output results/external_v093

In [ ]:
import json
from pathlib import Path

report = json.loads(
    Path("results/external_v093/report.json").read_text()
)

required = {
    "all_gates_pass": True,
    "validated_ODE_claimed": True,
    "ODE_existence_certified": True,
    "ODE_uniqueness_certified": True,
    "exact_response_preservation_certified": True,
    "uniform_L6_descent_certified_for_validated_solution": True,
    "global_flow_claimed": False,
}

checks = {k: report.get(k) == v for k, v in required.items()}
checks["protocol_hash"] = (
    report["protocol_sha256"]
    == "6d0aaefabd71f1d2986515ed84673f0083ae90d0344b9a1e92d7697ac08d061a"
)
checks["generator_hash"] = (
    report["generator_source_sha256"]
    == "3be3e07146ff0e505f08bae7bd0ec7f2895955f2540647fea3278fdba51db79c"
)

print(json.dumps(checks, indent=2, sort_keys=True))
assert all(checks.values()), "FAIL-CLOSED: ODE certificate did not reproduce"
print("PASS: VALIDATED INTRINSIC RESPONSE-FIBRE ODE MICROSTEP REPRODUCED")